<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/marcoteran/ml/blob/master/notebooks/ml_treesensemblesgbdt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab" title="Abrir y ejecutar en Google Colaboratory"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/marcoteran/ml/blob/master/notebooks/ml_treesensemblesgbdt.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Abrir en Kaggle" title="Abrir y ejecutar en Kaggle"/></a>
  </td>
</table>

# Sesión 05: Decision Trees & Ensemble Learning
## Guía Completa

**Machine Learning**

**Profesor:** Marco Terán  
**Fecha:** 2025

[Website](http://marcoteran.github.io/),
[Github](https://github.com/marcoteran),
[LinkedIn](https://www.linkedin.com/in/marcoteran/).
___

# 📊 Decision Trees & Ensemble Learning: Una Guía Completa

## Resumen Ejecutivo

Este notebook explora dos familias fundamentales de algoritmos de machine learning:

1. **Decision Trees**: Modelos interpretables que toman decisiones mediante reglas jerárquicas
2. **Ensemble Learning**: Métodos que combinan múltiples modelos para mejorar predicciones

### ¿Por qué son importantes?

- **Versatilidad**: Funcionan para clasificación y regresión
- **Interpretabilidad**: Los árboles individuales son fáciles de visualizar y explicar
- **Performance**: Los ensembles (Random Forest, XGBoost) dominan competencias de Kaggle
- **No requieren scaling**: A diferencia de SVM o regresión logística
- **Manejan features categóricas**: Especialmente útil en datos reales

### Roadmap del Notebook

* Parte 1-2: Árboles de Decisión → Fundamentos y regularización
* Parte 3: Voting → Combinación simple de modelos
* Parte 4: Bagging/Random Forest → Reducción de varianza
* Parte 5-6: Boosting → Reducción de sesgo secuencial
* Parte 7: Librerías modernas → XGBoost, LightGBM, CatBoost
* Parte 8: Hyperparameter Tuning → Optimización práctica
---

Definimos primero unas librerías y funciones que vamos a usar a durante la sesión:

In [ ]:
# Imports necesarios
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import (load_iris, load_breast_cancer, make_classification, 
                               make_regression, make_moons)
from sklearn.model_selection import (train_test_split, cross_val_score, 
                                     learning_curve, GridSearchCV)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                               BaggingClassifier, AdaBoostClassifier,
                               GradientBoostingClassifier, VotingClassifier,
                               ExtraTreesClassifier)
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                              roc_curve, auc, mean_squared_error, r2_score)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("✓ Librerías importadas correctamente")
print("✓ Configuración lista")

# PARTE 1: Decision Trees desde Cero

## Fundamentos Teóricos

### ¿Qué es un Decision Tree?

Un árbol de decisión es un modelo que aprende reglas de decisión simples inferidas de las características de los datos. Imagina un diagrama de flujo donde cada nodo interno representa una "prueba" sobre un atributo, cada rama representa el resultado de la prueba, y cada hoja representa una etiqueta de clase.

### Intuición Geométrica

Mientras que la regresión logística crea un **límite de decisión lineal**, los árboles de decisión crean **límites de decisión rectangulares** (paralelos a los ejes). Cada división (split) en el árbol corresponde a una línea perpendicular a uno de los ejes de características.

### El Algoritmo CART (Classification and Regression Trees)

El algoritmo trabaja de forma **greedy** y **top-down**:

1. Comenzar con todo el dataset en la raíz
2. Para cada feature, encontrar el mejor threshold que divide los datos
3. Elegir el split que maximiza la "pureza" de los nodos hijos
4. Repetir recursivamente para cada nodo hijo
5. Detener cuando se alcanza un criterio (profundidad, pureza, etc.)

### Criterios de División

**Para Clasificación:**

- **Gini Impurity**: $Gini(S) = 1 - \sum_{i=1}^{C} p_i^2$
  - Mide la probabilidad de clasificar incorrectamente un elemento aleatorio
  - Valores: 0 (puro) a 0.5 (binario, 50-50)

- **Entropy**: $H(S) = -\sum_{i=1}^{C} p_i \log_2(p_i)$
  - Mide el desorden o incertidumbre
  - Basado en teoría de información
  - Valores: 0 (puro) a 1 (binario, 50-50)

**Para Regresión:**
- **MSE (Mean Squared Error)**: Minimiza la varianza en cada nodo

### Information Gain

El criterio de selección de splits:

$$IG(S, A) = H(S) - \sum_{v \in Values(A)} \frac{|S_v|}{|S|} H(S_v)$$

Elegimos el split que **maximiza** el Information Gain (reduce más la impureza).

---


## Dataset de Ejemplo

In [ ]:
# Cargar dataset Iris
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

print("Dataset Iris:")
print(f"  - Muestras: {X_iris.shape[0]}")
print(f"  - Features: {X_iris.shape[1]}")
print(f"  - Clases: {len(np.unique(y_iris))}")
print(f"\nNombres de features: {iris.feature_names}")
print(f"Nombres de clases: {iris.target_names}")

# Usar solo 2 features para visualización
X_simple = X_iris[:, [2, 3]]  # petal length, petal width
X_train, X_test, y_train, y_test = train_test_split(X_simple, y_iris, 
                                                      test_size=0.3, random_state=42)

## Entrenar Decision Tree Básico

In [ ]:
# Crear y entrenar árbol
tree_clf = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_clf.fit(X_train, y_train)

# Evaluar
train_acc = tree_clf.score(X_train, y_train)
test_acc = tree_clf.score(X_test, y_test)

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Número de hojas: {tree_clf.get_n_leaves()}")
print(f"Profundidad del árbol: {tree_clf.get_depth()}")

## Visualizar el Árbol

In [ ]:
# Visualizar estructura del árbol
plt.figure(figsize=(20, 10))
plot_tree(tree_clf, 
          feature_names=['petal length', 'petal width'],
          class_names=iris.target_names,
          filled=True, 
          rounded=True,
          fontsize=12)
plt.title('Decision Tree - Iris Dataset', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Información de nodos
print("\nInformación del árbol:")
print(f"Feature usado en raíz: {iris.feature_names[tree_clf.tree_.feature[0]]}")
print(f"Threshold en raíz: {tree_clf.tree_.threshold[0]:.2f}")

## Decision Boundary Visualization

In [ ]:
# Crear mesh para decision boundary
h = 0.02
x_min, x_max = X_simple[:, 0].min() - 0.5, X_simple[:, 0].max() + 0.5
y_min, y_max = X_simple[:, 1].min() - 0.5, X_simple[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

# Predicciones en el mesh
Z = tree_clf.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plotear
plt.figure(figsize=(12, 8))
plt.contourf(xx, yy, Z, alpha=0.4, cmap='RdYlBu')
scatter = plt.scatter(X_simple[:, 0], X_simple[:, 1], c=y_iris, 
                     s=50, edgecolors='black', linewidth=1.5, cmap='RdYlBu')
plt.colorbar(scatter, label='Clase')
plt.xlabel('Petal Length (cm)', fontsize=13)
plt.ylabel('Petal Width (cm)', fontsize=13)
plt.title('Decision Boundary - Decision Tree', fontsize=15, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 🔍 Interpretación de Resultados

### ¿Qué nos dice el árbol?

Observa cómo el árbol aprendió patrones naturales en los datos:

1. **Primera división**: La feature más discriminativa separa inmediatamente una clase
2. **Nodos internos**: Cada nivel refina la decisión con reglas adicionales
3. **Hojas**: Predicción final basada en la clase mayoritaria del nodo

### Ventajas vs Desventajas

**✅ Ventajas:**
- Fácil de entender y visualizar
- Requiere poca preparación de datos
- Maneja datos numéricos y categóricos
- Modela interacciones no lineales automáticamente

**❌ Desventajas:**
- **Alta varianza**: Pequeños cambios en datos → árboles muy diferentes
- **Overfitting**: Sin regularización, memorizan el training set
- **Límites de decisión restrictivos**: Solo paralelos a ejes
- **Inestabilidad**: Árboles muy sensibles a perturbaciones

> 💡 **Insight clave**: Las desventajas de los árboles individuales son precisamente lo que los ensembles solucionan.

---

## Gini vs Entropy: ¿Realmente importa?

### Diferencias Teóricas

**Gini Impurity:**
- Computacionalmente más eficiente (sin logaritmos)
- Tiende a aislar la clase más frecuente en su propia rama
- Preferido en sklearn por defecto

**Entropy:**
- Basado en teoría de información (Shannon)
- Tiende a producir árboles más balanceados
- Penaliza más fuertemente las impurezas

### En la Práctica

La diferencia es **típicamente mínima** (1-2% en accuracy). Factores más importantes:
- Profundidad del árbol
- Criterios de parada
- Balance del dataset

> 📚 **Recomendación**: Usa Gini por defecto. Prueba Entropy si el árbol parece muy desbalanceado.


## Comparar Gini vs Entropy

In [ ]:
# Entrenar con Gini
tree_gini = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=42)
tree_gini.fit(X_train, y_train)

# Entrenar con Entropy
tree_entropy = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)
tree_entropy.fit(X_train, y_train)

# Comparar
print("Comparación Gini vs Entropy:")
print(f"\nGini:")
print(f"  Train accuracy: {tree_gini.score(X_train, y_train):.4f}")
print(f"  Test accuracy: {tree_gini.score(X_test, y_test):.4f}")
print(f"  Número de hojas: {tree_gini.get_n_leaves()}")

print(f"\nEntropy:")
print(f"  Train accuracy: {tree_entropy.score(X_train, y_train):.4f}")
print(f"  Test accuracy: {tree_entropy.score(X_test, y_test):.4f}")
print(f"  Número de hojas: {tree_entropy.get_n_leaves()}")

# Visualizar lado a lado
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for ax, tree, title in zip(axes, [tree_gini, tree_entropy], ['Gini', 'Entropy']):
    Z = tree.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.4, cmap='RdYlBu')
    ax.scatter(X_simple[:, 0], X_simple[:, 1], c=y_iris, 
              s=50, edgecolors='black', linewidth=1.5, cmap='RdYlBu')
    ax.set_xlabel('Petal Length (cm)', fontsize=12)
    ax.set_ylabel('Petal Width (cm)', fontsize=12)
    ax.set_title(f'Criterion: {title}', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Efecto de la Profundidad (Overfitting)

In [ ]:
# Entrenar árboles con diferentes profundidades
depths = [1, 2, 3, 5, 10, None]
train_scores = []
test_scores = []

for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    train_scores.append(tree.score(X_train, y_train))
    test_scores.append(tree.score(X_test, y_test))

# Plotear resultados
plt.figure(figsize=(12, 6))
x_labels = [str(d) if d is not None else '∞' for d in depths]
x_pos = np.arange(len(depths))

plt.plot(x_pos, train_scores, 'o-', linewidth=2.5, markersize=10, 
         label='Training Accuracy', color='#3498db')
plt.plot(x_pos, test_scores, 's-', linewidth=2.5, markersize=10, 
         label='Test Accuracy', color='#e74c3c')

plt.xticks(x_pos, x_labels)
plt.xlabel('Max Depth', fontsize=13)
plt.ylabel('Accuracy', fontsize=13)
plt.title('Efecto de max_depth en Performance', fontsize=15, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.ylim(0.8, 1.05)
plt.tight_layout()
plt.show()

print("\nTabla de resultados:")
results_df = pd.DataFrame({
    'max_depth': x_labels,
    'train_acc': train_scores,
    'test_acc': test_scores,
    'gap': np.array(train_scores) - np.array(test_scores)
})
print(results_df.to_string(index=False))

## Regression Trees

In [ ]:
# Generar datos para regresión
np.random.seed(42)
X_reg = np.sort(5 * np.random.rand(200, 1), axis=0)
y_reg = np.sin(X_reg).ravel() + np.random.randn(200) * 0.2

# Entrenar regression trees con diferentes profundidades
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
depths = [2, 5, 20]

for ax, depth in zip(axes, depths):
    tree_reg = DecisionTreeRegressor(max_depth=depth, random_state=42)
    tree_reg.fit(X_reg, y_reg)
    
    # Predicciones
    X_test_reg = np.arange(0, 5, 0.01)[:, np.newaxis]
    y_pred = tree_reg.predict(X_test_reg)
    
    # Plotear
    ax.scatter(X_reg, y_reg, s=20, edgecolor="black", c="darkorange", 
              label="Data", alpha=0.7)
    ax.plot(X_test_reg, y_pred, color="cornflowerblue", 
           label="Prediction", linewidth=3)
    ax.plot(X_test_reg, np.sin(X_test_reg), color="green", 
           label="True function", linewidth=2, linestyle='--', alpha=0.7)
    
    ax.set_xlabel("x", fontsize=12)
    ax.set_ylabel("y", fontsize=12)
    ax.set_title(f"max_depth = {depth}", fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Regression Trees: Efecto de la Profundidad', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

---
# PARTE 2: Regularización y Pruning

## 🛡️ El Problema del Overfitting en Árboles

### ¿Por qué los árboles sobreeajustan tan fácilmente?

Un árbol sin restricciones puede crear una hoja para **cada punto de entrenamiento**, memorizando completamente los datos. Esto es como:
- Estudiar solo ejemplos pasados de un examen (training data)
- Sin entender los conceptos subyacentes (patterns generalizables)
- Fallar en problemas ligeramente diferentes (test data)

### Estrategias de Regularización

#### 1️⃣ **Pre-Pruning (Detención temprana)**

Detener el crecimiento del árbol **durante** la construcción:

- `max_depth`: Profundidad máxima del árbol
  - Árboles poco profundos: underfitting (demasiado simple)
  - Árboles muy profundos: overfitting (demasiado complejo)
  
- `min_samples_split`: Mínimo de muestras para dividir un nodo
  - Valores altos: más conservador, previene splits de poca evidencia
  
- `min_samples_leaf`: Mínimo de muestras en cada hoja
  - Fuerza a que las predicciones tengan suficiente soporte estadístico

- `max_features`: Número de features a considerar por split
  - Introduce randomización (útil en Random Forest)

#### 2️⃣ **Post-Pruning (Cost-Complexity Pruning)**

Construir árbol completo, luego **podar** ramas que aportan poco:

$$R_\alpha(T) = R(T) + \alpha |T_{leaves}|$$

Donde:
- $R(T)$: Error del árbol
- $|T_{leaves}|$: Número de hojas
- $\alpha$ (ccp_alpha): Penalización por complejidad

> 🎯 **Trade-off fundamental**: Complejidad vs Error. Queremos el árbol más simple que generaliza bien.

## Hiperparámetros de Regularización

In [ ]:
# Dataset más complejo
X, y = make_moons(n_samples=500, noise=0.3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Configuraciones de regularización
configs = [
    {'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1},
    {'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 1},
    {'max_depth': None, 'min_samples_split': 20, 'min_samples_leaf': 10},
]

titles = ['Sin Regularización', 'max_depth=5', 'min_samples_split=20, min_samples_leaf=10']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, config, title in zip(axes, configs, titles):
    tree = DecisionTreeClassifier(**config, random_state=42)
    tree.fit(X_train, y_train)
    
    # Decision boundary
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = tree.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.4, cmap='RdYlBu')
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, 
              edgecolors='black', s=50, cmap='RdYlBu', alpha=0.8)
    
    train_acc = tree.score(X_train, y_train)
    test_acc = tree.score(X_test, y_test)
    
    ax.set_title(f'{title}\nTrain: {train_acc:.3f} | Test: {test_acc:.3f}',
                fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature 1', fontsize=11)
    if ax == axes[0]:
        ax.set_ylabel('Feature 2', fontsize=11)

plt.suptitle('Efecto de Regularización en Decision Trees', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## Cost Complexity Pruning (Alpha)

In [ ]:
# Extraer valores de alpha y encontrar mejor
path = tree_clf.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas
impurities = path.impurities

# Entrenar árboles para cada alpha
trees = []
train_scores_alpha = []
test_scores_alpha = []

for ccp_alpha in ccp_alphas[:-1]:  # Excluir último que es trivial
    tree = DecisionTreeClassifier(ccp_alpha=ccp_alpha, random_state=42)
    tree.fit(X_train, y_train)
    trees.append(tree)
    train_scores_alpha.append(tree.score(X_train, y_train))
    test_scores_alpha.append(tree.score(X_test, y_test))

# Plotear resultados
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Impureza vs alpha
axes[0].plot(ccp_alphas[:-1], impurities[:-1], marker='o', linewidth=2)
axes[0].set_xlabel('Alpha', fontsize=12)
axes[0].set_ylabel('Impureza Total', fontsize=12)
axes[0].set_title('Impureza vs Alpha', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Accuracy vs alpha
axes[1].plot(ccp_alphas[:-1], train_scores_alpha, marker='o', 
            linewidth=2, label='Train', color='#3498db')
axes[1].plot(ccp_alphas[:-1], test_scores_alpha, marker='s', 
            linewidth=2, label='Test', color='#e74c3c')
axes[1].set_xlabel('Alpha', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Accuracy vs Alpha (Post-Pruning)', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Encontrar mejor alpha
best_idx = np.argmax(test_scores_alpha)
best_alpha = ccp_alphas[best_idx]
best_tree = trees[best_idx]

print(f"\nMejor alpha: {best_alpha:.6f}")
print(f"Test accuracy con mejor alpha: {test_scores_alpha[best_idx]:.4f}")
print(f"Número de hojas: {best_tree.get_n_leaves()}")

---
# PARTE 3: Ensemble Learning - Voting

## La Sabiduría de las Multitudes

### Principio Fundamental

> "Ninguno de nosotros es tan inteligente como todos nosotros juntos"

Si entrenas múltiples modelos diferentes y combinas sus predicciones, el resultado suele superar cualquier modelo individual.

### ¿Por qué funciona?

#### Error de un Modelo

$$Error = Bias^2 + Variance + Irreducible Error$$

- **Bias**: Error por supuestos incorrectos del modelo
- **Variance**: Sensibilidad a fluctuaciones en training data
- **Irreducible Error**: Ruido inherente en los datos

#### Efecto del Ensemble

Diferentes modelos cometen **errores diferentes**:
- Un árbol se confunde con ciertas muestras
- Una SVM se confunde con otras muestras
- Al promediar, los errores se cancelan parcialmente

### Tipos de Voting

#### Hard Voting (Voto por mayoría)

$$\hat{y} = mode(\{h_1(x), h_2(x), ..., h_n(x)\})$$

- Cada modelo "vota" por una clase
- La clase con más votos gana
- Similar a una democracia directa

#### Soft Voting (Promedio de probabilidades)

$$\hat{y} = argmax_c \left(\frac{1}{n}\sum_{i=1}^{n} p_{i,c}\right)$$

- Considera la confianza de cada predicción
- Más robusto cuando un modelo es muy seguro
- Requiere que los modelos calculen probabilidades

> 💡 **Regla práctica**: Usa soft voting cuando sea posible. Aprovecha más información.

### Condiciones para un Buen Ensemble

1. **Diversidad**: Modelos deben ser diferentes
2. **Performance razonable**: Cada modelo debe ser mejor que azar
3. **Independencia**: Errores no correlacionados

## Voting Classifier

In [ ]:
# Dataset para clasificación
breast = load_breast_cancer()
X_breast = breast.data
y_breast = breast.target

X_train_bc, X_test_bc, y_train_bc, y_test_bc = train_test_split(
    X_breast, y_breast, test_size=0.3, random_state=42, stratify=y_breast
)

# Normalizar datos
scaler = StandardScaler()
X_train_bc_scaled = scaler.fit_transform(X_train_bc)
X_test_bc_scaled = scaler.transform(X_test_bc)

In [ ]:
# Importar modelos adicionales
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

# Crear modelos base
log_clf = LogisticRegression(max_iter=1000, random_state=42)
svm_clf = SVC(probability=True, random_state=42)
tree_clf = DecisionTreeClassifier(max_depth=10, random_state=42)
nb_clf = GaussianNB()

# Entrenar modelos individuales
models = {
    'Logistic Regression': log_clf,
    'SVM': svm_clf,
    'Decision Tree': tree_clf,
    'Naive Bayes': nb_clf
}

print("Performance de modelos individuales:")
print("-" * 60)

individual_scores = {}
for name, model in models.items():
    model.fit(X_train_bc_scaled, y_train_bc)
    score = model.score(X_test_bc_scaled, y_test_bc)
    individual_scores[name] = score
    print(f"{name:20s}: {score:.4f}")

In [ ]:
# Importar modelos adicionales
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

from sklearn.ensemble import VotingClassifier

# Voting Classifiers
hard_voting_clf = VotingClassifier(
    estimators=[
        ('lr', log_clf),
        ('svm', svm_clf),
        ('dt', tree_clf),
        ('nb', nb_clf)
    ],
    voting='hard'
)

soft_voting_clf = VotingClassifier(
    estimators=[
        ('lr', log_clf),
        ('svm', svm_clf),
        ('dt', tree_clf),
        ('nb', nb_clf)
    ],
    voting='soft'
)

# Entrenar y evaluar ensembles
hard_voting_clf.fit(X_train_bc_scaled, y_train_bc)
soft_voting_clf.fit(X_train_bc_scaled, y_train_bc)

hard_score = hard_voting_clf.score(X_test_bc_scaled, y_test_bc)
soft_score = soft_voting_clf.score(X_test_bc_scaled, y_test_bc)

print("\nPerformance de Voting Classifiers:")
print("-" * 60)
print(f"{'Hard Voting':20s}: {hard_score:.4f}")
print(f"{'Soft Voting':20s}: {soft_score:.4f}")

In [ ]:
# Visualizar comparación
all_scores = list(individual_scores.values()) + [hard_score, soft_score]
all_names = list(individual_scores.keys()) + ['Hard Voting', 'Soft Voting']
colors = ['steelblue']*4 + ['coral', 'gold']

plt.figure(figsize=(12, 6))
bars = plt.barh(all_names, all_scores, color=colors, edgecolor='black', linewidth=2)

# Anotar valores
for bar, score in zip(bars, all_scores):
    plt.text(score + 0.005, bar.get_y() + bar.get_height()/2, 
            f'{score:.4f}', va='center', fontsize=11, fontweight='bold')

plt.xlabel('Accuracy', fontsize=13)
plt.title('Comparación: Modelos Individuales vs Voting Ensembles', 
         fontsize=15, fontweight='bold')
plt.xlim(0.92, 0.99)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

---
# PARTE 4: Bagging y Random Forest

## 🎒 Bootstrap Aggregating (Bagging)

### La Intuición

Imagina que quieres saber la altura promedio de estudiantes en tu universidad:
- **Opción A**: Medir UNA muestra de 100 estudiantes
- **Opción B**: Tomar 10 muestras de 100 estudiantes y promediar

La Opción B es más confiable porque **reduce la varianza** del estimado.

### El Algoritmo

1. Crear $n$ **bootstrap samples** del training set
   - Bootstrap = muestreo con reemplazo
   - Cada sample tiene el mismo tamaño que el original
   - ~63% de datos únicos, ~37% duplicados

2. Entrenar un modelo en cada bootstrap sample
3. Combinar predicciones mediante:
   - **Clasificación**: Voting
   - **Regresión**: Promedio

### Bootstrap: El Concepto Clave

$$D_i = sample\_with\_replacement(D, |D|)$$

Al muestrear con reemplazo:
- Algunos ejemplos aparecen múltiples veces
- Otros no aparecen (Out-of-Bag)
- Crea diversidad sin necesidad de datos adicionales

### Out-of-Bag (OOB) Evaluation

En promedio, 37% de los datos no se usan en cada bootstrap:
- Podemos usarlos para validación
- Cada árbol es evaluado en sus propios OOB samples
- **OOB score** es un estimado de test performance sin necesidad de validation set

$$OOB\_score \approx Test\_accuracy$$

> 🎯 **Ventaja práctica**: Estimado de generalización "gratis" durante entrenamiento.

---

## 🌲 Random Forest: Bagging++

### ¿Por qué los árboles en Bagging son similares?

Problema: Si una feature es muy fuerte, **todos los árboles** la usarán en la raíz.
- Resultado: Árboles correlacionados
- Menos diversidad → menos beneficio del ensemble

### La Innovación de Random Forest

**Randomización adicional en cada split:**

1. Seleccionar subset aleatorio de $m$ features
2. Encontrar el mejor split solo dentro de esas $m$ features
3. Repetir en cada nodo

Típicamente: $m = \sqrt{p}$ para clasificación, $m = p/3$ para regresión

### Efecto de la Randomización

$$Correlation \downarrow \Rightarrow Variance_{ensemble} \downarrow$$

La randomización de features:
- **Decorrelaciona** los árboles
- Fuerza a usar features secundarias
- Descubre patrones que features dominantes ocultan

### Random Forest vs Single Tree

| Aspecto | Single Tree | Random Forest |
|---------|------------|---------------|
| Varianza | Alta | Baja (averaging) |
| Bias | Bajo | Ligeramente mayor |
| Interpretabilidad | Alta | Media (feature importance) |
| Training time | Rápido | Más lento |
| Prediction time | Muy rápido | Rápido (paralelo) |

> 📊 **En la práctica**: Random Forest es uno de los mejores algoritmos "out-of-the-box"

## Bagging desde Scikit-learn

In [ ]:
# Dataset
X, y = make_moons(n_samples=500, noise=0.3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Single Decision Tree
tree_single = DecisionTreeClassifier(random_state=42)
tree_single.fit(X_train, y_train)
single_score = tree_single.score(X_test, y_test)

# Bagging con Decision Trees
bagging_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=100,
    max_samples=1.0,
    bootstrap=True,
    oob_score=True,
    random_state=42
)
bagging_clf.fit(X_train, y_train)
bagging_score = bagging_clf.score(X_test, y_test)
oob_score = bagging_clf.oob_score_

print("Comparación:")
print(f"Single Tree:     {single_score:.4f}")
print(f"Bagging (100):   {bagging_score:.4f}")
print(f"OOB Score:       {oob_score:.4f}")
print(f"\nMejora: {bagging_score - single_score:.4f}")

In [ ]:
# Visualizar decision boundaries
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

h = 0.02
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

for ax, clf, title in zip(axes, [tree_single, bagging_clf], 
                          ['Single Decision Tree', 'Bagging (100 trees)']):
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.4, cmap='RdYlBu')
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, 
              edgecolors='black', s=50, cmap='RdYlBu', alpha=0.8)
    
    score = clf.score(X_test, y_test)
    ax.set_title(f'{title}\nTest Accuracy: {score:.4f}', 
                fontsize=13, fontweight='bold')
    ax.set_xlabel('Feature 1', fontsize=11)
    ax.set_ylabel('Feature 2', fontsize=11)

plt.tight_layout()
plt.show()

## Random Forest

In [ ]:
# Random Forest
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    bootstrap=True,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)
rf_clf.fit(X_train, y_train)
rf_score = rf_clf.score(X_test, y_test)
rf_oob_score = rf_clf.oob_score_

print("Random Forest Performance:")
print(f"Test Accuracy:  {rf_score:.4f}")
print(f"OOB Score:      {rf_oob_score:.4f}")
print(f"Número de árboles: {rf_clf.n_estimators}")

In [ ]:
# Comparar con diferentes números de árboles
n_estimators_range = [1, 5, 10, 25, 50, 100, 200, 500]
train_scores_rf = []
test_scores_rf = []
oob_scores_rf = []

for n_est in n_estimators_range:
    rf = RandomForestClassifier(n_estimators=n_est, oob_score=True, 
                                 random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    train_scores_rf.append(rf.score(X_train, y_train))
    test_scores_rf.append(rf.score(X_test, y_test))
    oob_scores_rf.append(rf.oob_score_)

# Plotear
plt.figure(figsize=(12, 6))
plt.plot(n_estimators_range, train_scores_rf, 'o-', linewidth=2.5, 
        label='Train', color='#3498db', markersize=8)
plt.plot(n_estimators_range, test_scores_rf, 's-', linewidth=2.5, 
        label='Test', color='#e74c3c', markersize=8)
plt.plot(n_estimators_range, oob_scores_rf, '^-', linewidth=2.5, 
        label='OOB', color='#2ecc71', markersize=8)

plt.xscale('log')
plt.xlabel('Número de Árboles', fontsize=13)
plt.ylabel('Accuracy', fontsize=13)
plt.title('Random Forest: Convergencia con Número de Árboles', 
         fontsize=15, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Feature Importance

In [ ]:
# Usar dataset completo de Iris
rf_full = RandomForestClassifier(n_estimators=200, random_state=42)
rf_full.fit(X_iris, y_iris)

# Extraer importancias
importances = rf_full.feature_importances_
indices = np.argsort(importances)[::-1]

print("Feature Importances:")
print("-" * 50)
for i, idx in enumerate(indices):
    print(f"{i+1}. {iris.feature_names[idx]:25s}: {importances[idx]:.4f}")

# Visualizar
plt.figure(figsize=(10, 6))
colors = plt.cm.viridis(importances[indices] / importances.max())
bars = plt.barh(range(len(importances)), importances[indices], 
               color=colors, edgecolor='black', linewidth=2)

plt.yticks(range(len(importances)), [iris.feature_names[i] for i in indices])
plt.xlabel('Importancia (Gini Decrease)', fontsize=13)
plt.title('Random Forest: Feature Importance', fontsize=15, fontweight='bold')
plt.grid(axis='x', alpha=0.3)

# Anotar valores
for bar, imp in zip(bars, importances[indices]):
    plt.text(imp + 0.01, bar.get_y() + bar.get_height()/2, 
            f'{imp:.3f}', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## Extra Trees

In [ ]:
# Extra Trees Classifier
et_clf = ExtraTreesClassifier(
    n_estimators=100,
    max_features='sqrt',
    bootstrap=False,  # Extra Trees no usa bootstrap por defecto
    random_state=42,
    n_jobs=-1
)
et_clf.fit(X_train, y_train)
et_score = et_clf.score(X_test, y_test)

print("Comparación Random Forest vs Extra Trees:")
print("-" * 50)
print(f"Random Forest:  {rf_score:.4f}")
print(f"Extra Trees:    {et_score:.4f}")
print(f"Diferencia:     {abs(rf_score - et_score):.4f}")

---
# PARTE 5: Boosting - AdaBoost

## 🚀 Un Paradigma Diferente: Boosting

### Bagging vs Boosting

| Bagging | Boosting |
|---------|----------|
| Modelos en **paralelo** | Modelos en **secuencia** |
| Reduce **varianza** | Reduce **bias** |
| Muestras aleatorias | Muestras ponderadas |
| Modelos independientes | Cada modelo corrige al anterior |

### La Idea Central

> "Aprende de tus errores"

1. Entrena modelo inicial
2. Identifica ejemplos mal clasificados
3. **Incrementa su peso** para el siguiente modelo
4. El nuevo modelo se enfoca en ejemplos difíciles
5. Repetir

### AdaBoost (Adaptive Boosting)

#### El Algoritmo Paso a Paso

**Inicialización:**
$$w_i^{(1)} = \frac{1}{N} \quad \forall i$$

Cada ejemplo comienza con peso igual.

**Para cada iteración $t = 1...T$:**

1. **Entrenar** weak learner $h_t$ con pesos $w^{(t)}$

2. **Calcular error ponderado:**
   $$\epsilon_t = \frac{\sum_{i: h_t(x_i) \neq y_i} w_i^{(t)}}{\sum_i w_i^{(t)}}$$

3. **Calcular peso del modelo** (confianza):
   $$\alpha_t = \frac{1}{2} \ln\left(\frac{1 - \epsilon_t}{\epsilon_t}\right)$$
   
   - Modelo perfecto ($\epsilon=0$): $\alpha \to \infty$
   - Modelo aleatorio ($\epsilon=0.5$): $\alpha = 0$

4. **Actualizar pesos de ejemplos:**
   $$w_i^{(t+1)} = w_i^{(t)} \cdot e^{-\alpha_t y_i h_t(x_i)}$$
   
   - Correctamente clasificados: peso disminuye
   - Incorrectamente clasificados: peso aumenta

**Predicción final:**
$$H(x) = sign\left(\sum_{t=1}^{T} \alpha_t h_t(x)\right)$$

### Weak Learners

AdaBoost típicamente usa **stumps** (árboles de profundidad 1):
- Cada árbol es apenas mejor que azar
- Pero la combinación es poderosa
- "Weak learners, strong ensemble"

> ⚠️ **Atención**: AdaBoost es sensible a outliers y ruido (los pondera cada vez más).

## AdaBoost Implementation

In [ ]:
# AdaBoost
ada_clf = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # stumps
    n_estimators=100,
    learning_rate=1.0,
    random_state=42
)
ada_clf.fit(X_train, y_train)
ada_score = ada_clf.score(X_test, y_test)

print(f"AdaBoost Test Accuracy: {ada_score:.4f}")
print(f"Número de estimadores: {ada_clf.n_estimators}")

In [ ]:
# Comparar con diferentes números de estimadores
n_estimators_ada = [1, 5, 10, 25, 50, 100, 200]
train_scores_ada = []
test_scores_ada = []

for n_est in n_estimators_ada:
    ada = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=n_est,
        learning_rate=1.0,
        random_state=42
    )
    ada.fit(X_train, y_train)
    train_scores_ada.append(ada.score(X_train, y_train))
    test_scores_ada.append(ada.score(X_test, y_test))

# Plotear
plt.figure(figsize=(12, 6))
plt.plot(n_estimators_ada, train_scores_ada, 'o-', linewidth=2.5, 
        label='Train', color='#3498db', markersize=8)
plt.plot(n_estimators_ada, test_scores_ada, 's-', linewidth=2.5, 
        label='Test', color='#e74c3c', markersize=8)

plt.xlabel('Número de Estimadores', fontsize=13)
plt.ylabel('Accuracy', fontsize=13)
plt.title('AdaBoost: Curvas de Aprendizaje', fontsize=15, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualizar evolución de pesos (simulado conceptualmente)
# Extraer estimadores y pesos
estimator_weights = ada_clf.estimator_weights_
estimator_errors = ada_clf.estimator_errors_

# Plotear
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Pesos
axes[0].plot(range(1, len(estimator_weights)+1), estimator_weights, 
            'o-', linewidth=2, markersize=6, color='#e74c3c')
axes[0].set_xlabel('Iteración', fontsize=12)
axes[0].set_ylabel('Peso del Estimador (α)', fontsize=12)
axes[0].set_title('AdaBoost: Pesos de Estimadores', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Errores
axes[1].plot(range(1, len(estimator_errors)+1), estimator_errors, 
            's-', linewidth=2, markersize=6, color='#3498db')
axes[1].axhline(0.5, color='red', linestyle='--', linewidth=2, alpha=0.7, 
               label='Random guessing')
axes[1].set_xlabel('Iteración', fontsize=12)
axes[1].set_ylabel('Error Ponderado', fontsize=12)
axes[1].set_title('AdaBoost: Errores de Estimadores', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nPrimeros 10 pesos de estimadores (α):")
print(estimator_weights[:10])
print(f"\nPrimeros 10 errores ponderados:")
print(estimator_errors[:10])

---
# PARTE 6: Gradient Boosting

## Generalización del Boosting

### De AdaBoost a Gradient Boosting

**AdaBoost**: Ajusta pesos de ejemplos
**Gradient Boosting**: Ajusta residuos directamente

### La Intuición

Imagina predecir precios de casas:

1. **Modelo 1** predice: "Casa vale $300K"
   - Valor real: $350K
   - **Residuo**: +$50K

2. **Modelo 2** aprende a predecir el residuo: "+$45K"
   - Nueva predicción: $300K + $45K = $345K
   - Nuevo residuo: +$5K

3. **Modelo 3** aprende: "+$4K"
   - Predicción final: $349K

Cada modelo **corrige los errores** del ensemble anterior.

### El Algoritmo

**Inicializar** con predicción constante:
$$F_0(x) = \arg\min_\gamma \sum_{i=1}^{n} L(y_i, \gamma)$$

**Para $m = 1...M$:**

1. **Calcular pseudo-residuos** (gradiente negativo):
   $$r_{im} = -\left[\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\right]_{F=F_{m-1}}$$

2. **Entrenar** árbol $h_m$ para predecir $r_{im}$

3. **Actualizar** ensemble:
   $$F_m(x) = F_{m-1}(x) + \nu \cdot h_m(x)$$
   
   donde $\nu$ es el **learning rate**

### Learning Rate: El Hiperparámetro Clave

$$F_m = F_{m-1} + \nu \cdot h_m$$

- **$\nu$ grande** (e.g., 1.0):
  - Aprende rápido
  - Menos iteraciones necesarias
  - Más propenso a overfitting

- **$\nu$ pequeño** (e.g., 0.01):
  - Aprende lentamente
  - Necesita más árboles
  - Mejor generalización

> 🎯 **Trade-off**: $\nu \downarrow$ requiere $M \uparrow$ pero mejora test performance

### Regularización en Gradient Boosting

1. **Shrinkage** (learning rate): $\nu \in (0, 1]$
2. **Tree constraints**: max_depth, min_samples_leaf
3. **Subsampling**: Entrenar cada árbol en subsample aleatorio
4. **Early stopping**: Monitorear validation error

### ¿Por qué funciona tan bien?

1. **Optimización directa** de la loss function
2. **Reduce bias** progresivamente
3. **Flexible**: Funciona con cualquier loss diferenciable
4. **Regularización** múltiple previene overfitting

## Gradient Boosting Classifier

In [ ]:
# Gradient Boosting
gb_clf = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=1.0,
    random_state=42
)
gb_clf.fit(X_train, y_train)
gb_score = gb_clf.score(X_test, y_test)

print(f"Gradient Boosting Test Accuracy: {gb_score:.4f}")

In [ ]:
# Staged predictions para ver evolución
train_scores_staged = []
test_scores_staged = []

for train_pred, test_pred in zip(gb_clf.staged_predict(X_train), 
                                  gb_clf.staged_predict(X_test)):
    train_scores_staged.append(accuracy_score(y_train, train_pred))
    test_scores_staged.append(accuracy_score(y_test, test_pred))

# Plotear
plt.figure(figsize=(12, 6))
plt.plot(range(1, len(train_scores_staged)+1), train_scores_staged, 
        linewidth=2.5, label='Train', color='#3498db')
plt.plot(range(1, len(test_scores_staged)+1), test_scores_staged, 
        linewidth=2.5, label='Test', color='#e74c3c')

# Marcar óptimo
best_iter = np.argmax(test_scores_staged) + 1
best_score = max(test_scores_staged)
plt.axvline(best_iter, color='green', linestyle='--', linewidth=2, alpha=0.7)
plt.scatter([best_iter], [best_score], s=200, color='green', 
           zorder=5, edgecolors='black', linewidth=2)
plt.text(best_iter + 2, best_score, 
        f'Óptimo: {best_iter} árboles\nAccuracy: {best_score:.4f}',
        fontsize=11, fontweight='bold', color='green',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

plt.xlabel('Número de Árboles', fontsize=13)
plt.ylabel('Accuracy', fontsize=13)
plt.title('Gradient Boosting: Curvas de Aprendizaje (Early Stopping)', 
         fontsize=15, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nMejor número de iteraciones: {best_iter}")
print(f"Mejor test accuracy: {best_score:.4f}")

## Efecto del Learning Rate

In [ ]:
# Comparar diferentes learning rates
learning_rates = [0.01, 0.05, 0.1, 0.5, 1.0]
colors_lr = plt.cm.viridis(np.linspace(0, 1, len(learning_rates)))

plt.figure(figsize=(14, 7))

for lr, color in zip(learning_rates, colors_lr):
    gb = GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=lr,
        max_depth=3,
        random_state=42
    )
    gb.fit(X_train, y_train)
    
    # Curva de test
    test_scores = []
    for test_pred in gb.staged_predict(X_test):
        test_scores.append(accuracy_score(y_test, test_pred))
    
    plt.plot(range(1, len(test_scores)+1), test_scores, 
            linewidth=2.5, label=f'LR = {lr}', color=color)

plt.xlabel('Número de Árboles', fontsize=13)
plt.ylabel('Test Accuracy', fontsize=13)
plt.title('Gradient Boosting: Efecto del Learning Rate', 
         fontsize=15, fontweight='bold')
plt.legend(fontsize=11, loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Feature Importance en Gradient Boosting

In [ ]:
# Entrenar en dataset completo
gb_full = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_full.fit(X_iris, y_iris)

# Feature importance
importances_gb = gb_full.feature_importances_
indices_gb = np.argsort(importances_gb)[::-1]

print("Feature Importances (Gradient Boosting):")
print("-" * 50)
for i, idx in enumerate(indices_gb):
    print(f"{i+1}. {iris.feature_names[idx]:25s}: {importances_gb[idx]:.4f}")

# Visualizar
plt.figure(figsize=(10, 6))
plt.barh(range(len(importances_gb)), importances_gb[indices_gb], 
        color='coral', edgecolor='black', linewidth=2)
plt.yticks(range(len(importances_gb)), [iris.feature_names[i] for i in indices_gb])
plt.xlabel('Importancia', fontsize=13)
plt.title('Gradient Boosting: Feature Importance', fontsize=15, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

---
# PARTE 7: Librerías Modernas de Boosting

## La Evolución de Gradient Boosting

### ¿Por qué necesitamos estas librerías?

**Sklearn GradientBoosting** es simple pero:
- Lento en datasets grandes
- Limitado en features categóricas
- Sin optimizaciones avanzadas

Las librerías modernas revolucionaron ML aplicado:
- Dominan competencias de Kaggle
- Usadas en producción (Google, Microsoft, Yandex)
- Órdenes de magnitud más rápidas

---

## GBoost (Extreme Gradient Boosting)

### Innovaciones Clave

#### 1️⃣ **Regularización en la Loss Function**

$$\mathcal{L} = \sum_{i=1}^{n} l(y_i, \hat{y}_i) + \sum_{k=1}^{K} \Omega(f_k)$$

donde:
$$\Omega(f) = \gamma T + \frac{1}{2}\lambda \sum_{j=1}^{T} w_j^2$$

- $\gamma$: Penalización por número de hojas
- $\lambda$: Regularización L2 en pesos de hojas

#### 2️⃣ **Aproximación de Segundo Orden**

Usa expansión de Taylor de segundo orden:
- Más preciso que solo primer orden (gradiente)
- Converge más rápido

#### 3️⃣ **Construcción de Árboles**

- **Exact Algorithm**: Para datasets pequeños
- **Approximate Algorithm**: Histogramas para datasets grandes
- **Sparsity-aware**: Maneja valores faltantes eficientemente

#### 4️⃣ **Paralelización**

- Construcción paralela de árboles
- Cache-aware access patterns
- Out-of-core computing

### Cuándo usar XGBoost

✅ Datasets tabulares medianos a grandes
✅ Competencias (excelente performance)
✅ Cuando necesitas interpretabilidad (SHAP values)
✅ Benchmark contra otros modelos

In [ ]:
# Instalar si es necesario: pip install xgboost
try:
    import xgboost as xgb
    import matplotlib.pyplot as plt
    from sklearn.metrics import accuracy_score

    # Preparar datos
    dtrain = xgb.DMatrix(X_train_bc_scaled, label=y_train_bc)
    dtest  = xgb.DMatrix(X_test_bc_scaled,  label=y_test_bc)

    # Parámetros
    params = {
        'max_depth': 3,
        'eta': 0.1,  # learning rate
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'seed': 42
    }

    # Entrenar
    evals = [(dtrain, 'train'), (dtest, 'test')]
    evals_result = {}  # <- necesario para capturar el historial
    xgb_model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=200,
        evals=evals,
        early_stopping_rounds=10,
        evals_result=evals_result,  # <- aquí se guarda el historial
        verbose_eval=False
    )

    # Predicciones
    y_pred_xgb = (xgb_model.predict(dtest) > 0.5).astype(int)
    xgb_acc = accuracy_score(y_test_bc, y_pred_xgb)

    print("XGBoost Results:")
    print(f"  Test Accuracy: {xgb_acc:.4f}")
    best_iter = getattr(xgb_model, "best_iteration", None)
    if best_iter is None:
        # si no hubo early stopping, usar el último índice entrenado
        best_iter = len(evals_result['train']['logloss']) - 1
    print(f"  Best iteration: {best_iter}")

    # Plotear learning curves
    train_logloss = evals_result['train']['logloss']
    test_logloss  = evals_result['test']['logloss']
    epochs = len(train_logloss)
    x_axis = range(epochs)

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(x_axis, train_logloss, linewidth=2.5, label='Train')
    ax.plot(x_axis, test_logloss,  linewidth=2.5, label='Test')
    ax.axvline(best_iter, linestyle='--', linewidth=2, alpha=0.7, label='Early Stopping')
    ax.set_xlabel('Número de Árboles', fontsize=13)
    ax.set_ylabel('Log Loss', fontsize=13)
    ax.set_title('XGBoost: Curvas de Aprendizaje', fontsize=15, fontweight='bold')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Feature importance
    importance_dict = xgb_model.get_score(importance_type='gain')
    print("\nTop 10 Features (XGBoost):")
    if importance_dict:
        sorted_features = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)[:10]
        for feat, score in sorted_features:
            print(f"  {feat}: {score:.2f}")
    else:
        print("  (No hay importancias disponibles con el tipo='gain')")

except ImportError:
    print("XGBoost no está instalado. Instalar con: pip install xgboost")


---

## LightGBM (Microsoft)

### La Gran Innovación: Leaf-wise Growth

**Nivel-wise (tradicional)**:
    O
   / \
  O   O
 /|   |\
O O   O O

**Leaf-wise (LightGBM)**:
      O
     / \
    O   O
   /|
  O O
 /|
O O

Expande la hoja con **mayor ganancia**, no por niveles.

### Ventajas

1. **Velocidad**: 10-20x más rápido que XGBoost
2. **Memoria**: Histograms reducen uso de RAM
3. **Accuracy**: Frecuentemente mejor que XGBoost

### Técnicas Especiales

#### Gradient-based One-Side Sampling (GOSS)
- Mantiene ejemplos con gradientes grandes
- Muestrea aleatoriamente gradientes pequeños
- **Menos datos → más rápido sin perder accuracy**

#### Exclusive Feature Bundling (EFB)
- Features mutuamente exclusivas se agrupan
- Reduce dimensionalidad efectiva

### Cuándo usar LightGBM

✅ **Datasets GRANDES** (>100K filas)
✅ Cuando la velocidad es crítica
✅ Features de alta dimensionalidad
✅ Memoria limitada


In [ ]:
# Instalar si es necesario: pip install lightgbm
try:
    import lightgbm as lgb
    import matplotlib.pyplot as plt
    from sklearn.metrics import accuracy_score

    # Crear datasets
    train_data = lgb.Dataset(X_train_bc_scaled, label=y_train_bc)
    test_data  = lgb.Dataset(X_test_bc_scaled,  label=y_test_bc, reference=train_data)

    # Parámetros
    params_lgb = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'num_leaves': 31,
        'learning_rate': 0.1,
        'feature_fraction': 0.9,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1,
        'seed': 42
    }

    # Entrenar
    evals_result = {}
    lgb_model = lgb.train(
        params=params_lgb,
        train_set=train_data,
        num_boost_round=200,
        valid_sets=[train_data, test_data],
        valid_names=['train', 'test'],
        callbacks=[
            lgb.early_stopping(stopping_rounds=10, verbose=False),
            lgb.log_evaluation(period=0),
            lgb.record_evaluation(evals_result)  # <- guarda el historial
        ]
    )

    # Predicciones
    y_pred_lgb = (lgb_model.predict(X_test_bc_scaled) > 0.5).astype(int)
    lgb_acc = accuracy_score(y_test_bc, y_pred_lgb)

    print("\nLightGBM Results:")
    print(f"  Test Accuracy: {lgb_acc:.4f}")
    best_iter = getattr(lgb_model, "best_iteration", None)
    if best_iter is None:
        # si no hubo early stopping, usar el último árbol entrenado
        best_iter = len(evals_result['train']['binary_logloss'])
    print(f"  Best iteration: {best_iter}")

    # Plotear curvas de aprendizaje
    train_loss = evals_result['train']['binary_logloss']
    test_loss  = evals_result['test']['binary_logloss']
    epochs = len(train_loss)
    x_axis = range(1, epochs + 1)

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(x_axis, train_loss, linewidth=2.5, label='Train')
    ax.plot(x_axis, test_loss,  linewidth=2.5, label='Test')
    # línea de early stopping (si aplica)
    if best_iter is not None and 1 <= best_iter <= epochs:
        ax.axvline(best_iter, linestyle='--', linewidth=2, alpha=0.7, label='Early Stopping')
    ax.set_xlabel('Número de Árboles', fontsize=13)
    ax.set_ylabel('Log Loss', fontsize=13)
    ax.set_title('LightGBM: Curvas de Aprendizaje', fontsize=15, fontweight='bold')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Feature importance (gain)
    importance_lgb = lgb_model.feature_importance(importance_type='gain')
    print("\nTop 10 Features (LightGBM):")
    if importance_lgb is not None and len(importance_lgb) > 0:
        feature_importance = sorted(
            zip(range(len(importance_lgb)), importance_lgb),
            key=lambda x: x[1],
            reverse=True
        )[:10]
        for idx, score in feature_importance:
            print(f"  Feature {idx}: {score:.2f}")
    else:
        print("  (No hay importancias disponibles con importance_type='gain')")

except ImportError:
    print("LightGBM no está instalado. Instalar con: pip install lightgbm")


---

## 🎯 CatBoost (Yandex)

### Especialización: Features Categóricas

### Ordered Target Statistics

Problema con one-hot encoding:
- Alta dimensionalidad
- No captura ordinality

Problema con label encoding:
- Valores arbitrarios
- Target leakage

**Solución de CatBoost:**

Para cada categoría, calcula estadística basada en target:
$$\hat{x}_k = \frac{\sum_{j=1}^{k-1} \mathbb{1}_{x_j = x_k} \cdot y_j + a \cdot P}{\sum_{j=1}^{k-1} \mathbb{1}_{x_j = x_k} + a}$$

- Usa solo ejemplos **anteriores** (previene leakage)
- Suavizado Bayesiano con prior $P$

### Ordered Boosting

Evita **prediction shift**:
- Diferentes permutaciones del dataset
- Cada modelo se entrena en subset diferente

### Defaults Inteligentes

CatBoost tiene los mejores defaults:
- Menos tuning necesario
- Robust out-of-the-box

### Cuándo usar CatBoost

✅ **Features categóricas** abundantes
✅ Quieres buenos resultados sin tuning
✅ Prevenir target leakage es crítico
✅ Prototipos rápidos

---

## 📊 Comparación Práctica

| Aspecto | XGBoost | LightGBM | CatBoost |
|---------|---------|----------|----------|
| **Velocidad** | Media | Muy rápida | Media |
| **Memoria** | Media | Baja | Alta |
| **Accuracy** | Alta | Alta | Muy Alta |
| **Cat. Features** | Manual | Manual | **Automático** |
| **Defaults** | Buenos | Buenos | **Excelentes** |
| **Comunidad** | Enorme | Grande | Mediana |
| **Documentación** | Excelente | Buena | Buena |

### Regla de Oro

1. **Prototipar** con CatBoost (mejores defaults)
2. **Optimizar** con LightGBM (velocidad)
3. **Producción** con XGBoost (estabilidad)

> 💡 En la práctica, las diferencias son pequeñas. La elección depende más del workflow que del performance.

In [ ]:
# Instalar si es necesario: pip install catboost
try:
    from catboost import CatBoostClassifier
    
    # Entrenar
    cat_model = CatBoostClassifier(
        iterations=200,
        learning_rate=0.1,
        depth=3,
        loss_function='Logloss',
        eval_metric='Accuracy',
        early_stopping_rounds=10,
        random_seed=42,
        verbose=False
    )
    
    cat_model.fit(
        X_train_bc_scaled, y_train_bc,
        eval_set=(X_test_bc_scaled, y_test_bc),
        plot=False
    )
    
    # Predicciones
    y_pred_cat = cat_model.predict(X_test_bc_scaled).flatten()
    cat_acc = accuracy_score(y_test_bc, y_pred_cat)
    
    print("\nCatBoost Results:")
    print(f"  Test Accuracy: {cat_acc:.4f}")
    print(f"  Best iteration: {cat_model.get_best_iteration()}")
    
    # Plotear learning curves
    train_scores_cat = cat_model.evals_result_['learn']['Accuracy']
    test_scores_cat = cat_model.evals_result_['validation']['Accuracy']
    
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(range(1, len(train_scores_cat)+1), train_scores_cat, 
           linewidth=2.5, label='Train')
    ax.plot(range(1, len(test_scores_cat)+1), test_scores_cat, 
           linewidth=2.5, label='Test')
    ax.set_xlabel('Número de Árboles', fontsize=13)
    ax.set_ylabel('Accuracy', fontsize=13)
    ax.set_title('CatBoost: Curvas de Aprendizaje', fontsize=15, fontweight='bold')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Feature importance
    importance_cat = cat_model.get_feature_importance()
    print("\nTop 10 Features (CatBoost):")
    feature_importance_cat = sorted(enumerate(importance_cat), 
                                    key=lambda x: x[1], reverse=True)[:10]
    for idx, score in feature_importance_cat:
        print(f"  Feature {idx}: {score:.2f}")
        
except ImportError:
    print("CatBoost no está instalado. Instalar con: pip install catboost")

## Comparación Final de Todos los Modelos

In [ ]:
# === Resumen y ranking de modelos (robusto y consistente con el dataset Breast Cancer) ===
from sklearn.metrics import accuracy_score
import numpy as np
import matplotlib.pyplot as plt

def safe_score(estimator, X, y, name):
    """Devuelve accuracy si el estimador está entrenado y las dimensiones coinciden; en caso contrario, None."""
    try:
        # Verificar que el estimador fue ajustado
        _ = getattr(estimator, "n_features_in_", None)
        # Verificar compatibilidad de features si aplica
        if _ is not None and X.shape[1] != estimator.n_features_in_:
            # Intentar detectar si el estimador se entrenó con un conjunto distinto (p.ej., 2 features)
            return None
        y_pred = estimator.predict(X)
        return accuracy_score(y, y_pred)
    except Exception:
        return None

results_summary = {}

# --- Decision Tree ---
# Usa el árbol entrenado sobre Breast Cancer si existe; intenta varios alias comunes
dt_candidate = None
for alias in ("tree_clf", "tree_single", "dt_clf", "decision_tree_clf"):
    if alias in globals():
        dt_candidate = globals()[alias]
        break

dt_score = None
if dt_candidate is not None:
    dt_score = safe_score(dt_candidate, X_test_bc_scaled, y_test_bc, "Decision Tree")
if dt_score is not None:
    results_summary["Decision Tree"] = dt_score

# --- Random Forest ---
if "rf_score" in globals() and isinstance(rf_score, (int, float)):
    results_summary["Random Forest"] = rf_score
elif "rf_clf" in globals():
    s = safe_score(rf_clf, X_test_bc_scaled, y_test_bc, "Random Forest")
    if s is not None:
        results_summary["Random Forest"] = s

# --- AdaBoost ---
if "ada_score" in globals() and isinstance(ada_score, (int, float)):
    results_summary["AdaBoost"] = ada_score
elif "ada_clf" in globals():
    s = safe_score(ada_clf, X_test_bc_scaled, y_test_bc, "AdaBoost")
    if s is not None:
        results_summary["AdaBoost"] = s

# --- Gradient Boosting ---
if "gb_score" in globals() and isinstance(gb_score, (int, float)):
    results_summary["Gradient Boosting"] = gb_score
elif "gb_clf" in globals():
    s = safe_score(gb_clf, X_test_bc_scaled, y_test_bc, "Gradient Boosting")
    if s is not None:
        results_summary["Gradient Boosting"] = s

# --- XGBoost / LightGBM / CatBoost (si están disponibles) ---
for name, var in [("XGBoost", "xgb_acc"), ("LightGBM", "lgb_acc"), ("CatBoost", "cat_acc")]:
    if var in globals():
        val = globals()[var]
        if isinstance(val, (int, float)):
            results_summary[name] = float(val)

# Si no hay resultados, evita crash
if not results_summary:
    print("\nNo hay resultados disponibles para rankear. Verifica que entrenaste los modelos con X_train_bc_scaled / y_train_bc.")
else:
    # Ordenar por performance
    sorted_results = sorted(results_summary.items(), key=lambda x: x[1], reverse=True)

    print("\n" + "="*70)
    print("RANKING FINAL DE MODELOS")
    print("="*70)
    for rank, (model, score) in enumerate(sorted_results, 1):
        print(f"{rank}. {model:20s}: {score:.4f}")

    # Visualizar
    models = [m for m, _ in sorted_results]
    scores = [s for _, s in sorted_results]

    plt.figure(figsize=(12, 8))
    colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(models)))
    bars = plt.barh(models, scores, color=colors, edgecolor='black', linewidth=2)

    # Anotar valores de forma segura (evitar salirse del eje x)
    for bar, score in zip(bars, scores):
        x_text = min(score + 0.01, 0.98)
        plt.text(x_text, bar.get_y() + bar.get_height()/2, f'{score:.4f}', va='center', fontsize=12)

    plt.xlabel('Test Accuracy', fontsize=13)
    plt.title('Comparación Final: Todos los Modelos', fontsize=16)
    left = max(min(scores) - 0.02, 0.0)
    plt.xlim(left, 1.0)
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()


---
# PARTE 8: HYPERPARAMETER TUNING
## Grid Search en Random Forest

In [ ]:
# Definir grid
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

# Grid Search
rf_gs = RandomForestClassifier(random_state=42, n_jobs=-1)
grid_search = GridSearchCV(
    rf_gs,
    param_grid_rf,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print("Iniciando Grid Search (esto puede tomar varios minutos)...")
grid_search.fit(X_train_bc_scaled, y_train_bc)

print("\nMejores hiperparámetros:")
print(grid_search.best_params_)
print(f"\nMejor score de CV: {grid_search.best_score_:.4f}")
print(f"Test score: {grid_search.score(X_test_bc_scaled, y_test_bc):.4f}")

In [ ]:
# Visualizar resultados del grid search
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df.sort_values('rank_test_score')

print("\nTop 10 configuraciones:")
print(results_df[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']].head(10))

# Plot top 20 configs
top_20 = results_df.head(20)
plt.figure(figsize=(12, 8))
plt.barh(range(len(top_20)), top_20['mean_test_score'], 
        xerr=top_20['std_test_score'], color='steelblue', 
        edgecolor='black', linewidth=1.5)
plt.yticks(range(len(top_20)), [f"Config {i+1}" for i in range(len(top_20))])
plt.xlabel('Mean CV Score', fontsize=13)
plt.title('Top 20 Configuraciones - Grid Search', fontsize=15, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# CONCLUSIONES Y MEJORES PRÁCTICAS

## Lecciones Fundamentales

### Jerarquía de Algoritmos

```
Single Decision Tree (interpretable, alta varianza)
    ↓
Bagging/Random Forest (reduce varianza mediante promediado)
    ↓
AdaBoost (reduce bias mediante ponderación adaptativa)
    ↓
Gradient Boosting (reduce bias mediante optimización de residuos)
    ↓
XGBoost/LightGBM/CatBoost (implementaciones optimizadas para producción)
```

### Decision Framework Práctico

**Usa Decision Trees cuando:**
- La interpretabilidad es crítica (explicar decisiones a stakeholders)
- Dataset pequeño (< 1000 muestras)
- Necesitas un prototipo rápido para exploración inicial
- Las reglas de decisión son naturalmente jerárquicas

**Usa Random Forest cuando:**
- Quieres buen performance sin ajuste exhaustivo de hiperparámetros
- La reducción de overfitting es prioritaria
- Necesitas estimados robustos de feature importance
- Tienes features ruidosas o irrelevantes

**Usa Gradient Boosting (sklearn) cuando:**
- El performance predictivo es crítico
- Puedes invertir tiempo en hyperparameter tuning
- Dataset mediano (1K-100K muestras)
- Necesitas control fino del proceso de entrenamiento

**Usa XGBoost/LightGBM/CatBoost cuando:**
- Competencia de ML o despliegue en producción
- Datasets grandes (> 100K muestras)
- Múltiples features categóricas (prioriza CatBoost)
- La velocidad de entrenamiento es importante (prioriza LightGBM)

---

## Comparación de Trade-offs

| Aspecto | Single Tree | Random Forest | Gradient Boosting | XGBoost/LightGBM |
|---------|------------|---------------|-------------------|------------------|
| **Interpretabilidad** | Muy Alta | Media | Baja | Baja |
| **Velocidad Training** | Muy Rápida | Media | Lenta | Rápida |
| **Velocidad Predicción** | Muy Rápida | Rápida | Media | Rápida |
| **Overfitting** | Alto riesgo | Bajo riesgo | Medio riesgo | Bajo riesgo |
| **Tuning requerido** | Mínimo | Mínimo | Extensivo | Moderado |
| **Paralelización** | No aplica | Excelente | Limitada | Excelente |

---

## Pipeline Recomendado para Proyectos Reales

### Fase 1: Baseline (Día 1)
1. **Random Forest con defaults**: Establece baseline rápido
2. **Análisis de feature importance**: Identifica features clave
3. **Validación cruzada**: Estima performance real

### Fase 2: Optimización (Días 2-3)
1. **Gradient Boosting**: Prueba sklearn GradientBoosting
2. **Hyperparameter search**: Grid/Random search en parámetros clave
   - `max_depth`: [3, 5, 7]
   - `learning_rate`: [0.01, 0.05, 0.1]
   - `n_estimators`: Usa early stopping
3. **Cross-validation**: 5-fold CV para estabilidad

### Fase 3: Refinamiento (Días 4-5)
1. **XGBoost/LightGBM/CatBoost**: Implementaciones optimizadas
2. **Feature engineering**: Basado en importances del Fase 2
3. **Ensemble de mejores modelos**: Voting o stacking

### Fase 4: Validación Final
1. **Hold-out test set**: Nunca tocado hasta este punto
2. **Análisis de errores**: Entender fallos del modelo
3. **Calibración de probabilidades**: Si se necesitan probabilidades confiables

---

## Mejores Prácticas por Algoritmo

### Decision Trees
- **Siempre regulariza**: Usa `max_depth` entre 3-10
- **Post-pruning**: Experimenta con `ccp_alpha` en [0.001, 0.1]
- **Visualiza**: Un árbol incomprensible es un árbol sobreajustado
- **Validación**: Compara train vs test accuracy para detectar overfitting

### Random Forest
- **Número de árboles**: Comienza con 100-200, aumenta si mejora OOB score
- **max_features**: `sqrt` para clasificación, `1/3` para regresión
- **Usa OOB score**: Validación gratuita durante entrenamiento
- **Paralelización**: Siempre usa `n_jobs=-1`
- **No podar árboles individuales**: El ensemble maneja la complejidad

### Gradient Boosting
- **Learning rate pequeño**: 0.01-0.1 típicamente
- **Árboles poco profundos**: `max_depth` 3-5 previene overfitting
- **Early stopping es obligatorio**: Monitorea validation set
- **Subsample < 1.0**: Introduce regularización estocástica (0.8 es común)
- **Trade-off fundamental**: `learning_rate * n_estimators` = constante

### XGBoost/LightGBM/CatBoost
- **Comienza con defaults**: Especialmente CatBoost tiene excelentes defaults
- **Early stopping rounds**: 10-50 dependiendo del dataset
- **Regularización**: Experimenta con `reg_alpha` (L1) y `reg_lambda` (L2)
- **Cross-validation integrado**: Usa CV nativo de la librería
- **Features categóricas**: En CatBoost, marca columnas categóricas explícitamente

---

## Errores Comunes a Evitar

### 1. No separar datos correctamente
- **Error**: Validar en datos de entrenamiento
- **Solución**: Train/validation/test split estricto

### 2. Olvidar early stopping en boosting
- **Error**: Entrenar número fijo de iteraciones
- **Solución**: Monitorear validation loss y detener cuando deja de mejorar

### 3. Tuning prematuro
- **Error**: Optimizar hiperparámetros antes de entender el problema
- **Solución**: Baseline primero, luego optimiza

### 4. Ignorar data leakage
- **Error**: Features que contienen información del target
- **Solución**: Análisis temporal y validación cuidadosa

### 5. Sobre-interpretar feature importance
- **Error**: Asumir que importancia = causalidad
- **Solución**: Feature importance es correlacional, no causal

---

## Recursos para Profundizar

### Papers Fundamentales
1. **Breiman, L. (2001)** - "Random Forests" 
   - Paper original de Random Forest
   
2. **Friedman, J. H. (2001)** - "Greedy Function Approximation: A Gradient Boosting Machine"
   - Fundamentos matemáticos de Gradient Boosting
   
3. **Chen, T., & Guestrin, C. (2016)** - "XGBoost: A Scalable Tree Boosting System"
   - Arquitectura y optimizaciones de XGBoost

### Libros Recomendados
1. **"The Elements of Statistical Learning"** - Hastie, Tibshirani, Friedman
   - Capítulos 9, 10, 15: Tratamiento matemático riguroso
   
2. **"Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow"** - Aurélien Géron
   - Capítulos 6-7: Implementación práctica
   
3. **"Introduction to Statistical Learning"** - James, Witten, Hastie, Tibshirani
   - Versión más accesible de ESL

### Cursos Online
- **Stanford CS229**: Machine Learning (Andrew Ng)
- **Fast.ai**: Practical Deep Learning
- **Coursera**: Machine Learning Specialization

### Documentación Técnica
- Scikit-learn User Guide: Decision Trees y Ensemble Methods
- XGBoost Documentation: Parameters tuning
- LightGBM Documentation: Advanced features
- CatBoost Tutorials: Categorical features handling

---

## Palabras Finales

Los árboles de decisión y sus ensembles representan una de las familias más exitosas de algoritmos en machine learning aplicado. Su fortaleza radica en:

1. **Versatilidad**: Clasificación, regresión, ranking
2. **Robustez**: Pocos supuestos sobre la distribución de datos
3. **Performance**: Estado del arte en problemas tabulares
4. **Escalabilidad**: Implementaciones modernas manejan millones de muestras

El camino del aprendizaje es iterativo: comienza con árboles simples para entender los fundamentos, progresa a Random Forest para aplicaciones prácticas, y domina Gradient Boosting para competencias y producción. Cada nivel construye sobre el anterior, reforzando intuiciones y expandiendo tu toolkit.

La maestría viene no de memorizar fórmulas, sino de entender cuándo y por qué aplicar cada técnica. Este notebook es un punto de partida. La verdadera comprensión viene de experimentar con datos reales, cometer errores, y aprender de ellos.


In [ ]:
print("\n" + "="*70)
print("RESUMEN DE MEJORES PRÁCTICAS")
print("="*70)

practices = """
1. DECISION TREES:
   - Usar max_depth para evitar overfitting
   - Considerar min_samples_split y min_samples_leaf
   - Probar cost complexity pruning (ccp_alpha)
   - Visualizar el árbol para interpretabilidad

2. RANDOM FOREST:
   - Comenzar con 100-200 árboles
   - max_features='sqrt' para clasificación
   - Usar OOB score para validación rápida
   - n_jobs=-1 para paralelización

3. GRADIENT BOOSTING:
   - Learning rate pequeño (0.01-0.1) con más árboles
   - max_depth bajo (3-5) para evitar overfitting
   - SIEMPRE usar early stopping
   - Subsample < 1.0 para regularización

4. XGBOOST/LIGHTGBM/CATBOOST:
   - XGBoost: Balance general, bien documentado
   - LightGBM: Datasets grandes, velocidad
   - CatBoost: Features categóricas, buenos defaults
   
5. GENERAL:
   - Cross-validation para evaluación robusta
   - Feature importance para interpretabilidad
   - Comparar múltiples algoritmos
   - Considerar ensemble de ensembles (stacking)
"""

print(practices)

print("="*70)
print("FIN DEL NOTEBOOK")
print("="*70)

# EJERCICIOS PROPUESTOS

In [ ]:
print("\n" + "="*70)
print("EJERCICIOS PARA PRACTICAR")
print("="*70)

exercises = """
1. Implementar Information Gain desde cero
2. Crear un árbol de decisión simple usando solo NumPy
3. Comparar Bagging con Pasting empíricamente
4. Implementar AdaBoost manualmente (5-10 iteraciones)
5. Crear un voting classifier con 5 modelos diferentes
6. Optimizar XGBoost usando Optuna/Hyperopt
7. Analizar feature importance en dataset real complejo
8. Comparar tiempo de entrenamiento de todos los algoritmos
9. Crear visualizaciones animadas del proceso de boosting
10. Implementar stacking ensemble con diferentes niveles
"""

print(exercises)

___
¡Todo bien! ¡Es todo por hoy! 😀